# Debugging the 8-Category `8cat_bondwire` Training Crash & Zero-Loss Collapse

**Purpose of this notebook:** a chronological, evidence-backed log of every debugging
step taken on 2026-07-17/18 while fine-tuning the new 8-category bondwire model
(`experiments/8cat_bondwire`). Each step below shows: what we saw, what we did about
it, and the actual result — so a future investigation (or you, reviewing this) doesn't
need to re-run multi-hour training jobs just to re-derive what's already been found.

**Where the real evidence lives (persists on HPC, doesn't need re-running):**
- Anomalous-batch dumps: `experiments/8cat_bondwire/dumps/anomalous_batches/*.pt`
- Auto-generated JPEG previews: `experiments/8cat_bondwire/dumps/anomalous_batches_jpeg/`
- Training logs: `experiments/8cat_bondwire/bsub_logs/train.<jobid>.{out,err}`
- All code changes are committed on branch `fix/loss-none-pair-crash`

**Current status (as of last check):** root cause of the zero-loss collapse is
**confirmed**: `backbone.vision_backbone.trunk.patch_embed.proj`'s Conv2d **weights**
go NaN, which cascades through the entire network, and the existing `nan_to_num`
loss guards (added earlier to stop a different crash) silently absorb the resulting
NaN loss into an exact `0.0` instead of crashing — so training silently "learns
nothing" for a while instead of dying loudly. The remaining open question is *why*
the weights go NaN in the first place (see Step 8 — in progress).


## Timeline at a glance

| Step | Job(s) | What happened | Fix / tool built |
|---|---|---|---|
| 1 | `237753606` | Crashed after ~8h at epoch5/step980: `TypeError: 'NoneType' object is not subscriptable` | Guarded `None` per-sample pair in `_count_matched_pairs` |
| 2 | `237892821` | Resumed fine, then hit a **persistent zero-loss** (500+ steps, starting ~step630-670) | Built anomalous-batch `.pt` dump + JPEG visualizer |
| 3 | (offline) | JPEG visualizer's box overlay landed in a blank padding region | Fixed per-image box/mask attribution (`num_boxes` is per image×category, not per image) |
| 4 | `237899477` | Zero-loss recurred (step 900+) | Added per-loss-term breakdown: **every one of ~30 terms was exactly 0.0** |
| 5 | `237906220` | Recurred again (step 730) | Captured actual model output at matched indices → **predictions were NaN**, not zero |
| 6 | `237913353` | The capture itself **caused a new OOM** at step 730 | Fixed: slice to matched-only rows, move to CPU immediately (was holding full GPU tensors) |
| 7 | `237927128` (bs=2, `SAM3_NAN_DEBUG_HOOKS=1`) | Recurred (step ~1240) | Forward hooks on 1167 submodules → **all 255 hits point to `patch_embed.proj`, 100% NaN output** |
| 8 | `237937642` (running) | — | Added a targeted weight-NaN + AMP-scaler-state check right after every optimizer step, to catch the *exact* step the corruption happens |

Also found and fixed along the way: a **second** crash (`IndexError` on an empty,
non-`None` per-sample indices entry) in the same diagnostic helper.


## Step 1 — Original crash: `TypeError: 'NoneType' object is not subscriptable`

**Job `237753606`** ran for ~8 hours (epoch 5/10, step 980/1808, loss healthy and
decreasing right up to the crash) then died with:

```
File "sam3/train/loss/sam3_loss.py", line 55, in _count_matched_pairs
    return sum(
File "sam3/train/loss/sam3_loss.py", line 56, in <genexpr>
    int(pair[0].numel()) if hasattr(pair[0], "numel") else len(pair[0])
                                    ~~~~^^^
TypeError: 'NoneType' object is not subscriptable
```

**Root cause:** `_count_matched_pairs()` (a diagnostic helper, itself added just the
day before to investigate a *different* issue) guarded the top-level `indices` being
`None`, but not an *individual* entry inside a list of per-sample tuples. That
individual entry is `None` for a **negative sample** — an image with zero matched
instances for the queried categories, which is expected and common since this
dataset uses `include_negatives=True`.

**Fix:** one-line guard — treat a `None` per-sample entry as 0 matched pairs, same
as the already-existing top-level guard.

**Result:** resubmitted as job `237892821`, confirmed it **resumed cleanly from the
epoch-5 checkpoint** and passed the exact step (980) that crashed before.


## Step 2 — A different failure mode surfaces: the "zero-loss collapse"

After resuming (job `237892821`), training ran fine for a while, then the logged
`train_all_loss` went to **exactly `0.00e+00`** and stayed there:

```
Train Epoch: [5][ 980/1808] | ... | Losses/train_all_loss: 0.00e+00 (5.89e+01) | ...
Train Epoch: [5][ 990/1808] | ... | Losses/train_all_loss: 0.00e+00 (5.83e+01) | ...
Train Epoch: [5][1000/1808] | ... | Losses/train_all_loss: 0.00e+00 (5.77e+01) | ...
... (500+ consecutive steps, no recovery before we stopped it)
```

This does **not** crash — `0.0` is a perfectly finite float, so
`math.isfinite(loss.item())` never trips. Training just silently continues, learning
nothing for every affected step. This exact pattern had been seen once before
(2026-07-17, `experiments/5cat_matrix_chip_v2`, epoch 9) and was left as an open,
"rare/hardware-driven" mystery at the time.

**Tooling built to investigate it (`sam3/train/trainer.py`,
`.github/scripts/inspect_anomalous_batch.py`):**
- `Trainer._maybe_dump_anomalous_batch()` — whenever a real (non-`DummyLoss`) loss
  goes non-finite **or exactly zero**, dumps the batch (post-augmentation images +
  targets, pre-augmentation raw source) to
  `experiments/8cat_bondwire/dumps/anomalous_batches/<phase>_<key>_epoch<E>_step<S>_<n>.pt`
  — rate-limited to 3 dumps per run.
- `.github/scripts/inspect_anomalous_batch.py <dump_path>` — prints a side-by-side
  pre- vs. post-augmentation comparison, to rule out a degenerate/corrupted source
  sample as the cause.


### Reusable tool: load and inspect any anomalous-batch dump

The cell below is a CPU-only, no-training-needed way to re-inspect any `.pt` dump
already sitting in `experiments/8cat_bondwire/dumps/anomalous_batches/`. It works
because this Windows path is the same network mount as the HPC path (see repo
memory) — no `ssh`/HPC job needed, just a Python kernel with `torch` + `PIL`
available locally, or connect this notebook to the HPC kernel via the
`hpc-jupyter-connect` skill if you don't have torch installed locally.

Change `DUMP_PATH` to any `.pt` file under `anomalous_batches/` and re-run.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent  # nbs/ -> repo root
sys.path.insert(0, str(REPO_ROOT / ".github" / "scripts"))

import torch
from inspect_anomalous_batch import (
    _describe_raw,
    _describe_post,
    _describe_model_output_debug,
    save_post_augmentation_jpegs,
)

# Change this to any dump you want to inspect:
DUMP_PATH = REPO_ROOT / "experiments/8cat_bondwire/dumps/anomalous_batches/train_all_epoch5_step9768_0.pt"

payload = torch.load(DUMP_PATH, map_location="cpu", weights_only=False)
print(f"phase={payload['phase']} key={payload['key']} epoch={payload['epoch']} "
      f"step={payload['step']} loss_value={payload['loss_value']}")

_describe_raw(payload.get("raw_images"), payload.get("raw_boxes"))
_describe_post(payload["img_batch"], payload["find_targets"])
_describe_model_output_debug(payload.get("model_output_debug"))


In [ ]:
# Optional: save post-augmentation JPEGs (box + mask overlays) from this same dump,
# to actually LOOK at the images without needing a training job.
OUT_DIR = REPO_ROOT / "experiments/8cat_bondwire/dumps/anomalous_batches_jpeg" / DUMP_PATH.stem
save_post_augmentation_jpegs(payload, str(OUT_DIR))
print(f"Saved JPEGs to: {OUT_DIR}")


## Step 3 — A real bug caught by actually looking at the JPEGs

The first version of `save_post_augmentation_jpegs()` rendered a box in a **blank
padding region** of an image — on an image that, per the raw pre-augmentation data,
actually had 14 real annotated objects elsewhere in frame.

**Root cause:** `num_boxes` has one entry per **(image, category)** query — e.g. 32
entries for a 4-image batch with `category_chunk_size=8` — **not** one entry per
image. The first version chunked the flat `boxes`/`segments` list directly by each
raw `num_boxes` entry, producing batch-size-many groups from the *wrong* slice
boundaries, silently misattributing boxes/masks to the wrong image.

**Fix:** sum every `len(num_boxes) / batch_size` consecutive entries into the
correct per-image total *first*, then chunk the flat list by those totals.

**Verified:** the corrected JPEG for the same image now shows the correct 14 boxes,
landing squarely on real chip die pads — matching the raw annotation count exactly.

This is also why the notebook's reusable inspection cell above is worth having:
this bug would have been very easy to miss without visually rendering the result.


## Step 4 — Breaking the total loss down per term: every single one is 0.0

`sam3_loss.py`'s `Sam3LossWrapper.compute_loss()` sums `total_core_loss` across ~6
"stages" (main output + auxiliary decoder layers + first-stage output) × 3 loss
functions (`Boxes`, `IABCEMdetr`, `Masks`) × o2o/o2m matching = ~30 individual
contributions. The existing diagnostic only logged the *aggregate* being zero. Added
per-term tracking so we could see exactly which ones were responsible:

```
(Ha) Zero total_core_loss diagnostic: num_boxes(normalized)=21.0 targets_num_boxes_sum(raw)=21
n_stages=6 matched_pairs_per_stage=[2, 2, 2, 2, 2, 2]
nonzero_contributions=NONE -- every single term was exactly 0.0
all_zero_terms=[('Boxes','o2o'), ('Boxes','o2m'), ('IABCEMdetr','o2o'), ('IABCEMdetr','o2m'),
                 ('Masks','o2o'), ('Masks','o2m'), ... (all 30 entries)]
```

**Why this matters:** every one of these loss terms (L1 box distance, GIoU,
focal/BCE classification, dice, mask focal) is **non-negative by construction** —
none of them can go negative to cancel another out. For the *sum* to be exactly
`0.0` while **real matched pairs exist** (`matched_pairs_per_stage=[2,2,2,2,2,2]`,
`num_boxes=21`), **every individual term must independently be exactly `0.0` at the
same time**. That's not a plausible coincidence across ~30 independent computations
— it points to one shared upstream cause, not a bug in one specific loss function.

At this point the leading theory was: matched pairs point at a zero-valued padding
slot on both prediction and target sides (trivially giving 0 loss). Step 5 tests
this directly.


## Step 5 — Capturing the actual model output: the padding-slot theory is WRONG

Added `Sam3LossWrapper.last_zero_loss_debug` (`sam3_loss.py`) — whenever the
zero-loss diagnostic fires, it stashes the main stage's `pred_boxes` /
`pred_logits` / `pred_masks` **at the matched indices** for `trainer.py`'s dump to
include. First real capture (`train_all_epoch5_step9768_0.pt`):

```python
pred_boxes[0,6]  = [nan, nan, nan, nan]
pred_logits[0,6] = [nan]
pred_masks[0,6]  min/max/mean = nan, nan, nan
```

**The matched predictions are literally `NaN` — not zero, and not a padding-slot
match.** That rules out the Step 4 theory. The real mechanism:

1. The model's forward pass produces `NaN` at some query position(s).
2. Every loss function consuming that query's prediction computes a `NaN` loss.
3. The `nan_to_num(nan=0.0, ...)` guards — added 2026-07-16 specifically to stop a
   *different*, earlier `FloatingPointError` crash — convert that `NaN` loss to
   exactly `0.0`.
4. Since **every** term across every stage is independently `0.0` this way, the
   summed total is `0.0` too, silently — `math.isfinite()` never trips, so training
   continues with a dead (zero-gradient) step instead of crashing.

**In short: the 2026-07-16 crash fix didn't fix the underlying bug — it hid it.**
The model still produces NaN under the same conditions; it just no longer tells you
when. This is documented in full in `.github/skills/loss-debugging/SKILL.md` Step 9.

You can see this exact evidence yourself by running the reusable inspection cell
above against `train_all_epoch5_step9768_0.pt` (or any other dump) —
`_describe_model_output_debug()` prints precisely these matched values.


## Step 6 — Own mistake: the debug capture itself caused an OOM

The *first* version of the `last_zero_loss_debug` capture kept the **full**
`pred_boxes`/`pred_logits`/`pred_masks` tensors (all ~200 queries, not just the
matched ones) GPU-resident via `.detach()` alone (no `.cpu()`), held in
`self.last_zero_loss_debug` until `trainer.py` consumed them — well after this
step's normal tensors would have been freed. `pred_masks` in particular is
`num_queries x H x W`, large.

This directly caused job `237913353` to **OOM-crash at step 730** — the exact same
moment a zero-loss event (and therefore this new capture) fired, on a config
already at ~44/44.39GB with zero memory headroom.

```
torch.OutOfMemoryError: CUDA out of memory. Tried to allocate 2.53 GiB.
GPU 0 has a total capacity of 44.39 GiB of which 1.91 GiB is free.
```

**Fix:** slice down to just the matched `(batch, query)` positions (a handful of
rows, not the whole tensor) and move to CPU **immediately at the capture site** in
`sam3_loss.py`, instead of leaving full-size GPU tensors alive across the boundary
into `trainer.py`. Field names changed accordingly:
`matched_pred_boxes` / `matched_pred_logits` / `matched_pred_masks` / `matched_pairs`.

**Lesson:** any debugging instrumentation added to an already memory-tight training
loop needs the same memory discipline as the training code itself — "it's just for
debugging" doesn't exempt it from causing real crashes.


## Step 7 — Bisecting the forward pass: it's `patch_embed.proj`

`model_output_debug` can only see the NaN *after* it already reached the final
prediction heads — it doesn't say *where in the forward pass* it originated. Added
`_install_nan_debug_hooks()` (`trainer.py`, gated behind `SAM3_NAN_DEBUG_HOOKS=1`,
off by default since checking every submodule's output has real overhead):
registers a forward hook on **every** named submodule (1167 of them for this
model) that checks the output for non-finite values, logging only the *first* (in
forward call order) submodule to produce NaN/Inf each step.

**Ran with this enabled** (job `237927128`, `train_batch_size` temporarily dropped
2→ to make room for the extra transient memory the hooks need — see the "gotcha"
note below) — the collapse recurred at step ~1240, and every one of **255 hits, with
zero exceptions, pointed at the exact same module**:

```
(Ha) NaN-debug: FIRST non-finite output at module=backbone.vision_backbone.trunk.patch_embed.proj
type=Conv2d shape=(2, 1024, 72, 72) n_nan=10616832 n_inf=0 forward_call=1083
```

`n_nan=10616832` = `2 x 1024 x 72 x 72` **exactly** — **100% of the output tensor
is NaN**, for both images in the batch. This is the **very first layer** of the
vision backbone (patch embedding, right after the input image).

**Interpretation:** if only some elements were NaN, or it varied which
image/module was affected over time, that would suggest a bad *input* image. But
100% of the output being NaN for a layer this early, with total consistency across
hundreds of steps and multiple dumps, points to something more specific: **the
`patch_embed.proj` layer's WEIGHTS themselves became NaN** at some earlier point —
not a per-batch numerical fluke in this layer's own computation. Once a Conv2d's
weights go NaN, every subsequent forward pass produces NaN from that point onward,
for the rest of training — exactly matching the observed *persistent, total*
collapse (not intermittent per-batch).

**Gotcha hit along the way:** the hooks' own `torch.isfinite(t).all()` calls
themselves need a small transient memory allocation. On a config already at
44.00/44.39GB with zero headroom, even that pushed it over
(`torch.OutOfMemoryError: ... Tried to allocate 648.00 MiB`), especially since
gradient checkpointing re-runs the forward pass during backward (recompute),
**doubling** hook firings. Had to drop `train_batch_size` from 4 → 2 *just for this
debug run* to create headroom — reverted back to 4 afterward for real training.


## Step 8 — CONFIRMED: caught the exact moment the weights corrupt

Traced the gradient-clipping / AMP order in `_train_epoch`:

```python
self.scaler.unscale_(self.optim.optimizer)   # records found_inf from the PRE-clip gradient
self.gradient_clipper(model=self.model)      # torch.nn.utils.clip_grad_norm_(..., max_norm=0.1)
...
self.scaler.step(self.optim.optimizer)       # should SKIP the update if found_inf was recorded
self.scaler.update()
```

This is the textbook-correct order — `unscale_()` should catch a non-finite
gradient *before* clipping touches the values, so `scaler.step()` should skip
applying **any** update when the original gradient was non-finite, regardless of
what `clip_grad_norm_` does to the (discarded) values afterward. In theory, this
should make it impossible for these weights to go NaN through a normal
"skipped-step" path.

Since the weights clearly did corrupt anyway, added
`_check_patch_embed_weight_nan()` (`trainer.py`): checks `patch_embed.proj`'s
weight/bias for non-finite values **immediately after every `scaler.step()` +
`scaler.update()`**, logging the exact step number and the scaler's current
`get_scale()` value the first time it's detected. This will tell us definitively:

- **Which exact step** the corruption happens at (not just "somewhere in this
  epoch").
- **What the AMP scaler's scale factor was** at that moment — helps confirm/refute
  whether the interaction between `bfloat16` training and `GradScaler` (originally
  designed for `float16`'s much narrower range) is implicated, since `bfloat16`
  doesn't strictly need loss scaling at all.
- Whether `scaler.step()` is failing to skip a non-finite-gradient update (a real,
  fixable bug/gap) or something else entirely is corrupting these specific weights
  (e.g. optimizer state (`exp_avg`/`exp_avg_sq` in AdamW) already being corrupted
  from an earlier step, which would keep producing NaN updates forever after even
  with clean future gradients, since it's a persistent exponential moving average
  that can't self-heal).

**Status: CONFIRMED.** Job `237937642` (resumed from epoch 5, `SAM3_NAN_DEBUG_HOOKS=1`,
`train_batch_size=2` for headroom) hit the weight-NaN check and logged the smoking gun:

```
WEIGHT-NaN: patch_embed.proj.weight went non-finite at step=9766,

scaler.get_scale()=2.938735877055719e-39, n_nan=602112, n_inf=0, param_numel=602112

```kind of numerical corruption it was meant to prevent.

into subnormal fp32 territory, at which point it starts *causing* the exact

`n_nan == param_numel` — **100% of the Conv2d weight's elements** went NaN in one— and over enough halvings, nothing stops the scale itself from spiraling down

step, and it happened at the *exact* moment `scaler.get_scale()` had decayed todynamically responds to any inf/nan by halving the scale (`backoff_factor=0.5`)

`2.94e-39` — a **subnormal fp32 value** (~143 consecutive `backoff_factor=0.5x`all. Because `GradScaler` doesn't know the dtype doesn't need it, it still

halvings from the default `init_scale=65536.0`). This is not "a bad gradient slippedwhich has the **same exponent range as fp32** and never needed loss scaling at

through" — the scale value *itself* had degenerated into fp32's subnormal range,gradient underflow. This training run uses `bfloat16` (`amp_dtype: bfloat16`),

where the `scale`/`unscale_` round-trip (`loss * scale`, later `grad / scale`) isexponent range, where dynamic loss scaling is genuinely necessary to prevent

numerically unstable in its own right, regardless of what the underlying gradient**Root cause:** `torch.amp.GradScaler` was designed for `float16`'s narrow ~5-bit

was.

## Second crash along the way: `IndexError` on an empty (non-`None`) indices entry

While attempting to run job `237927128` with the new hooks, hit a **different**
crash in the same `_count_matched_pairs()` diagnostic helper from Step 1:

```
IndexError: index 0 is out of bounds for dimension 0 with size 0
    (int(pair[0].numel()) if hasattr(pair[0], "numel") else len(pair[0]))
                    ~~~~^^^
```

**Root cause:** a per-sample `indices` entry can be a *genuinely empty* sequence
(`()`/`[]`) for a fully-negative sample — not `None` this time, so the Step 1 guard
didn't catch it. Indexing `pair[0]` on an empty `pair` fails.

**Fix:** check `len(pair) == 0` in addition to `pair is None`, and wrap each
per-sample count in a `try`/`except` so this diagnostic-only helper can never crash
the training loop again regardless of what shape `indices` takes in the future.


## Step 9 — The fix: pin the GradScaler scale for bfloat16

Added a config-controlled fix instead of unconditionally disabling `GradScaler`,
so it can be A/B tested and reverted via config alone:

**`sam3/train/configs/full_train_config.yaml`** — new field under `optim.amp`:

```yaml
optim:
  amp:
    enabled: True
    amp_dtype: bfloat16
    bypass_dynamic_scale_for_bf16: True   # default True — this IS the fix
```

**`sam3/train/trainer.py`** — `OptimAMPConf` dataclass gets the matching field
(`bypass_dynamic_scale_for_bf16: bool = True`), and the `self.scaler = torch.amp.GradScaler(...)`
construction becomes conditional:

```python
if bypass_dynamic_scale:  # amp.bypass_dynamic_scale_for_bf16 and amp_dtype is bf16
    self.scaler = torch.amp.GradScaler(
        self.device,
        enabled=self.optim_conf.amp.enabled if self.optim_conf else False,
        init_scale=1.0,
        growth_factor=1.0 + 1e-7,
        backoff_factor=1.0 - 1e-7,
        growth_interval=2**31 - 1,
    )
else:
    self.scaler = torch.amp.GradScaler(...)  # original dynamic-scale behavior
```

**Why not literally `growth_factor=1.0, backoff_factor=1.0`?** `torch.amp.GradScaler`'s
constructor hard-asserts `growth_factor > 1.0` and `backoff_factor < 1.0` — a literal
`1.0` raises `AssertionError` (hit this on the first resubmit attempt, job `237943473`,
instant `EXIT` after 6s). Used near-identity values instead (`1 ± 1e-7`) with an
astronomically large `growth_interval` (`2**31 - 1`) — growth practically never
triggers, and even unlimited consecutive backoffs only erode the scale by `~1e-7`
per event, so it would take an unrealistic number of them (vs. the ~143 halvings
@ `0.5x` that produced the `2.94e-39` subnormal collapse) to meaningfully move the
scale away from `init_scale=1.0`. Verified the constructor call succeeds locally
before resubmitting.

**Why keep `enabled=True` at all instead of just disabling the scaler for bf16?**
`GradScaler.step()`'s found-inf skip-step safety net (records `found_inf` in
`unscale_()`, skips the optimizer step in `step()` if set) stays active either way
— pinning the *scale value* removes the pathological decay without giving up the
protection against a genuinely non-finite gradient reaching the optimizer.

**Validation run:** killed the in-flight debug job `237937642` (`bkill`), amended
the commit, resubmitted as job `237943948` with the **same debug settings**
(`SAM3_NAN_DEBUG_HOOKS=1`, `train_batch_size=2`, resumed from the same epoch-5
checkpoint) so the fix could be directly compared against the exact same run that
produced the `237937642` collapse. Confirmed at startup:

```
INFO ... trainer.py:1528: (Ha) amp.bypass_dynamic_scale_for_bf16=True: pinning
GradScaler's scale at (effectively) a constant 1.0 instead of the default dynamic
scale, since bfloat16 doesn't need loss scaling and its decay was confirmed to
corrupt weights.
```

...and early training steps show healthy, non-zero, non-NaN losses. Long-running
validation (does the collapse recur over enough steps to pass where `237937642`
died at step 9766?) still in progress — see Open items below.


## Step 10 — First fix attempt FAILED on checkpoint resume (found immediately via validation)

The Step 9 fix was validated by resubmitting as job `237943948` — and the
collapse **recurred anyway**, with the *exact same* subnormal scale value as
before:

```
WEIGHT-NaN: patch_embed.proj.weight went non-finite at step=10261,
scaler.get_scale()=2.938735877055719e-39, n_nan=602112, n_inf=0, param_numel=602112
```

Byte-for-byte identical `2.938735877055719e-39` to the original `237937642`
collapse, at nearly the same relative step (~1220/3617 into epoch 5 both
times). That's not a coincidence — it means the Step 9 fix's pinned-scale
`GradScaler` construction was never actually taking effect at runtime.

**Root cause of the fix failing:** `_load_resuming_checkpoint()` (`trainer.py`)
unconditionally does:

```python
if self.optim_conf.amp.enabled and "scaler" in checkpoint:
    self.scaler.load_state_dict(checkpoint["scaler"])
```

`torch.amp.GradScaler.state_dict()`/`load_state_dict()` round-trips **all
five** internal fields: `scale`, `growth_factor`, `backoff_factor`,
`growth_interval`, `_growth_tracker`. Since this debug run always resumes
from the same epoch-5 checkpoint — saved by an earlier run using the
**original dynamic scaler** (`growth_factor=2.0`, `backoff_factor=0.5`, and
whatever scale that run had reached) — loading it silently overwrote the
freshly-constructed pinned-scale scaler with the old dynamic settings right
after `__init__`. The fix was 100% correct in isolation; it just never
survived the checkpoint-resume step this investigation happens to always go
through.

**Second fix:** stash the bypass flag as `self._scaler_bypass_dynamic_scale`
during `__init__`, and skip the `scaler.load_state_dict(...)` call entirely
in `_load_resuming_checkpoint()` when it's `True` — keeping the pinned ~1.0
scale intact across resumes instead of inheriting stale dynamic-scaler
state:

```python
if (
    self.optim_conf.amp.enabled
    and "scaler" in checkpoint
    and not getattr(self, "_scaler_bypass_dynamic_scale", False)
):
    self.scaler.load_state_dict(checkpoint["scaler"])
elif getattr(self, "_scaler_bypass_dynamic_scale", False):
    logging.info("(Ha) ... skipping restore of checkpoint's saved GradScaler state ...")
```

**Re-validation:** killed `237943948`, committed the fix, resubmitted as job
`237950477` from the SAME checkpoint/debug settings. Confirmed at startup —
**both** log lines now fire, in the right order:

```
INFO ... (Ha) amp.bypass_dynamic_scale_for_bf16=True: pinning GradScaler's scale
       at (effectively) a constant 1.0 ...
INFO ... (Ha) amp.bypass_dynamic_scale_for_bf16=True: skipping restore of
       checkpoint's saved GradScaler state (scale/growth_factor/backoff_factor)
       to keep the scale pinned at ~1.0 across resumes.
```

Early steps show clean, non-zero, non-NaN losses with zero `WEIGHT-NaN` hits
so far. Long-run validation (does it stay clean past step ~1220, where BOTH
prior attempts collapsed?) in progress — see Open Items below.

**Lesson for next time:** when a fix touches any object that gets restored
from a checkpoint (optimizer, scaler, scheduler, EMA, etc.), always check
whether `_load_resuming_checkpoint()`/`save_checkpoint()` round-trips that
object's state — a config-level fix to the constructor is silently
worthless if a later `load_state_dict()` call overwrites it. This cost one
full validation cycle (job `237943948`, ~50 min of run time) to discover.


## How to check on the current/next debugging run

All commands below run from the local machine (this repo's root), over SSH — no
need to open a new terminal session per check.

**Job/queue status:**
```bash
ssh hpc_muc 'bjobs -l <job_id>'
ssh hpc_muc 'bqueues | grep -E "QUEUE_NAME|^gpu"'
```

**Training progress + zero-loss / NaN-debug hit counts:**
```bash
ssh hpc_muc '
F=/home/hpc_storage/data/projects/userB/test_sam3/sam3_fine_tune/experiments/8cat_bondwire/bsub_logs/train.<job_id>.out
grep "Train Epoch" "$F" | tail -10
grep -c "train_all_loss: 0.00e+00" "$F"
grep -c "NaN-debug: FIRST" "$F"
grep "WEIGHT-NaN" "$F"
'
```

**List new anomalous-batch dumps (persist even after a job ends/crashes):**
```bash
ssh hpc_muc 'ls -la /home/hpc_storage/data/projects/userB/test_sam3/sam3_fine_tune/experiments/8cat_bondwire/dumps/anomalous_batches/'
```

Then use this notebook's reusable inspection cells (Step 2 section above) to load
and analyze any new dump **without needing another training run**.

## Environment variables / config used for debugging (not the real training run)

- `SAM3_NAN_DEBUG_HOOKS=1` — enables the forward-hook NaN localization (Step 7) and
  the weight-NaN check (Step 8). **Off by default** — real per-step overhead.
- `train_batch_size` was temporarily dropped `4 → 2` in
  `sam3/train/configs/full_train_config.yaml` only to make memory headroom for the
  hooks (Step 7's gotcha). **Must be reverted to `4` before the real production
  run** once this investigation concludes.
- `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` — already standard for every
  submit on this config (fragmentation mitigation, unrelated to this investigation).

## Open items / next steps

1. ✅ **DONE** — Root cause confirmed (Step 8): `scaler.get_scale()` decayed into
   fp32 subnormal territory (`2.94e-39`) and corrupted `patch_embed.proj`'s weights
   100% at step 9766.
2. ✅ **DONE** — First fix implemented (Step 9): `optim.amp.bypass_dynamic_scale_for_bf16`
   config flag (default `True`), pins `GradScaler`'s scale at ~1.0 for `bfloat16`
   training while keeping the found-inf skip-step safety net active.
3. ❌ **FAILED, then FIXED (Step 10)** — First validation run (job `237943948`)
   reproduced the IDENTICAL collapse/scale value, because
   `_load_resuming_checkpoint()` was unconditionally restoring the checkpoint's
   OLD dynamic-scaler state via `scaler.load_state_dict(...)`, silently
   overwriting the pinned scaler. Fixed by skipping that restore when
   `bypass_dynamic_scale_for_bf16` is active. Both fixes committed on
   `fix/loss-none-pair-crash`.
4. 🔄 **IN PROGRESS** — Re-validation run (job `237950477`, same debug
   settings/checkpoint as both prior attempts): confirm the collapse does NOT
   recur past step ~1220 (where BOTH prior runs died), and that
   `scaler.get_scale()` stays pinned near `1.0` for the duration (check via the
   commands below — no more `WEIGHT-NaN` log lines should appear).
5. ⏳ **NOT STARTED** — Once validated: revert `train_batch_size` back to `4` and
   unset `SAM3_NAN_DEBUG_HOOKS` before the next "real" (non-debugging) production
   training submission.
6. ⏳ **NOT STARTED** — After a full, clean production run: proceed to model
   evaluation (the original pending task before this investigation began).
7. Update this notebook with each new finding — keep it the single source of truth
   for this investigation instead of re-deriving from chat history.

**Quick validation check for job `237950477`:**
```bash
ssh hpc_muc '
F=/home/hpc_storage/data/projects/userB/test_sam3/sam3_fine_tune/experiments/8cat_bondwire/bsub_logs/train.237950477.out
grep "Train Epoch" "$F" | tail -10
grep -c "WEIGHT-NaN" "$F"          # should stay 0
grep -c "train_all_loss: 0.00e+00" "$F"   # should stay 0 (no zero-loss collapse)
'
```
